# PySpark drills 2: DataFrame essentials

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-drills-02-dataframe-essentials.ipynb)

Practice for [PySpark 2: DataFrame essentials](pyspark-02-dataframe-essentials.ipynb). Each exercise uses the `customers`, `orders` and `order_items` tables from the lesson.

Write your answer in the empty cell under each exercise, run it, and compare your output with the expected output.
The reference answer is hidden under the exercise; open it when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `customers` (one row per customer; customer 4 has no region, and 7 and 8 have never ordered), `orders` (one row per order; order 12 lands just after midnight UTC) and `order_items` (one row per order line; each order's lines add up to its amount): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#setup-and-sample-data).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F

customers = spark.createDataFrame([
    (1, "Acme Corp", "North", date(2024, 1, 3)),
    (2, "Bright Retail", "North", date(2024, 2, 11)),
    (3, "Cyan Traders", "South", date(2024, 3, 15)),
    (4, "Delta Grocers", None, date(2024, 5, 2)),  # region never arrived from the CRM
    (5, "Echo Supplies", "West", date(2024, 6, 20)),
    (6, "Foxtrot Foods", "West", date(2024, 8, 1)),
    (7, "Gamma Sports", "East", date(2024, 9, 14)),  # 7 and 8 have never ordered
    (8, "Hotel Homeware", "East", date(2024, 11, 5)),
], "customer_id int, name string, region string, signup_date date")

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),  # a bulk reorder
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

## Exercise 1: Recent large orders

**Difficulty:** Easy · about 5 min

Return paid or shipped orders with an amount over 100, newest first, with the columns `order_id`, `status` and
`amount`.

**Expected output**

| order_id | status | amount |
| --- | --- | --- |
| 9 | shipped | 1180.00 |
| 7 | paid | 310.00 |
| 6 | shipped | 149.00 |
| 4 | paid | 260.00 |
| 1 | paid | 120.00 |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    orders.where(F.col("status").isin("paid", "shipped") & (F.col("amount") > 100))
    .orderBy(F.col("order_ts").desc())
    .select("order_id", "status", "amount")
)
result.show()

## Exercise 2: Customers outside the North

**Difficulty:** Easy · about 5 min

Return every customer whose region is not `North`, **including** customers with no region, as `customer_id`, `name`
and `region`, ordered by `customer_id`.

**Expected output**

| customer_id | name | region |
| --- | --- | --- |
| 3 | Cyan Traders | South |
| 4 | Delta Grocers | null |
| 5 | Echo Supplies | West |
| 6 | Foxtrot Foods | West |
| 7 | Gamma Sports | East |
| 8 | Hotel Homeware | East |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    customers.where((F.col("region") != "North") | F.col("region").isNull())
    .select("customer_id", "name", "region")
    .orderBy("customer_id")
)
result.show()

## Exercise 3: Combine two deliveries

**Difficulty:** Medium · about 10 min

Two systems deliver order statuses with the same columns in a different order, and one order appears in both.
Combine them by column name, keep one copy of exact duplicates, and order by `order_id`.

**Sample input**

**`delivery_a`**

| order_id | status |
| --- | --- |
| 101 | paid |
| 102 | placed |

**`delivery_b`**

| status | order_id |
| --- | --- |
| placed | 102 |
| shipped | 103 |

**Expected output**

| order_id | status |
| --- | --- |
| 101 | paid |
| 102 | placed |
| 103 | shipped |

In [ ]:
# Sample data for this exercise.
delivery_a = spark.createDataFrame([(101, "paid"), (102, "placed")], "order_id int, status string")
delivery_b = spark.createDataFrame([("placed", 102), ("shipped", 103)], "status string, order_id int")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = delivery_a.unionByName(delivery_b).distinct().orderBy("order_id")
result.show()

## Exercise 4: Reconcile two lists of order ids

**Difficulty:** Medium · about 10 min

Compare the order ids in a source system and in the warehouse. Return two results, each with one column `order_id`
ordered ascending: `missing`, the distinct ids in the source but not in the warehouse, and `extra`, the distinct ids in
the warehouse but not in the source.

**Sample input**

**`source`**

| order_id |
| --- |
| 1 |
| 2 |
| 3 |
| 3 |
| 5 |

**`warehouse`**

| order_id |
| --- |
| 2 |
| 3 |
| 4 |

**Expected output**

| order_id |
| --- |
| 1 |
| 5 |

| order_id |
| --- |
| 4 |

In [ ]:
# Sample data for this exercise.
source = spark.createDataFrame([(1,), (2,), (3,), (3,), (5,)], "order_id int")
warehouse = spark.createDataFrame([(2,), (3,), (4,)], "order_id int")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
missing = source.subtract(warehouse).orderBy("order_id")
extra = warehouse.subtract(source).orderBy("order_id")
missing.show()
extra.show()

## Exercise 5: The three largest order lines

**Difficulty:** Medium · about 10 min

Compute `line_amount = qty * unit_price` for every order line and return the three largest lines with `order_id`,
`sku`, `qty` and `line_amount`. Break ties by `order_id`, so the output is the same on every run.

**Expected output**

| order_id | sku | qty | line_amount |
| --- | --- | --- | --- |
| 9 | SKU-4011 | 20 | 1180.00 |
| 11 | SKU-4013 | 1 | 210.00 |
| 7 | SKU-4006 | 1 | 189.00 |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    order_items.withColumn("line_amount", F.col("qty") * F.col("unit_price"))
    .orderBy(F.desc("line_amount"), "order_id")
    .select("order_id", "sku", "qty", "line_amount")
    .limit(3)
)
result.show()

**Next:** [PySpark 3: types, functions and nulls](pyspark-03-types-functions-and-nulls.ipynb)